# Parallel Mean-Cov Model

## Imports and Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import copy
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
import shap
import torch

from lightning.pytorch.utilities.warnings import PossibleUserWarning
from scipy import stats
from scipy.stats import wilcoxon
from sklearn.preprocessing import MinMaxScaler
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm

from matplotlib.colors import LinearSegmentedColormap

white_red = LinearSegmentedColormap.from_list(
    "white_to_shap_red",
    ["white", "#FFB3C7", "#FF0051"],
)
blue_red = LinearSegmentedColormap.from_list(
    "blue_to_red",
    ["#008BFB", "#FFFFFF", "#FF0051"],
)



from src.classes import Anscombe, DotDict

from src.helper_functions import (
    build_lit_model,
    compute_correlation,
    compute_group_influence_matrix,
    compute_group_metrics,
    compute_metrics,
    compute_mse,
    compute_r2,
    compute_shap_values,
    load_cache,
    predict_loader,
    predict_loader_group_conditionings,
    prepare_dataset,
    prepare_loader,
    save_cache,
    seed_everything,
    shuffle_dataset,
)

from src.plot_functions import (
    get_cov_loading_matrices,
    get_cov_length_scales,
    get_covariance_matrix,
    plot_unit_spikes,
    plot_unit_covariance,
    plot_cov_length_scales,
    plot_cov_loading_matrix,
    plot_cov_noise,
    plot_covariance_matrix,
    plot_group_influence_matrix,
    plot_model_metric_improvement,
    plot_model_metrics_comparison,
    plot_prediction,
    plot_shap,
    plot_shap_hist,
    plot_spike_summary,
    plot_train_valid_metrics_comparison,
    plot_training_history,
    plot_variable_selectivity_hist,
    plot_variable_selectivity_curves,
    plot_variable_selectivity_pca,
    plot_class_selectivity_pca,
    plot_variable_selectivity_matrix,
    plot_class_selectivity_matrix,
    save_shap_plot,
)

logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

warnings.filterwarnings("ignore", category=PossibleUserWarning)
warnings.filterwarnings("ignore", message=".*does not have many workers.*")
warnings.filterwarnings("ignore", message=r".*Checkpoint directory .* exists and is not empty.*", category=UserWarning, module=r"lightning\.pytorch\.callbacks\.model_checkpoint")

SEED = 1
seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [76]:
Conf = DotDict({
    "run_id": 1,
    "seed": SEED,
    "paths": {
        "data": "data.mat",
    },
    "session_idx": 15,
    "device": device,
    "data": {
        "start_time": -2,
        "end_time": 2,
    },
    "training": {
        "batch_size": 32,
        "max_epoch": 500,
        "min_delta": 1e-4,
        "patience": 10,
        "n_shuffles": 100,
        "n_permutations": 100,
    },
    "model_type": {
        "name": "mean",
        "mean": {
            "n_hidden": 16,
            "n_heads": 2,
            "n_layers": 1,
            "nonlinearity": "gelu",
            "dropout": 0,
        },
        "cov": {
            "n_latent": 16,
            "n_hidden": 2,
            "n_heads": 1,
            "n_layers": 1,
            "nonlinearity": "gelu",
            "dropout": 0,
        },
    },
    "optimization": {
        "Adam": {
            "lr": 0.01,
            "weight_decay": 0,
        },
        "Reduce": {
            "factor": 0.5,
            "patience":5,
            "min_lr": 1e-10,
        },
    }
})

## Data Preparation

In [77]:
data = sio.loadmat(Conf.paths.data, squeeze_me=True, struct_as_record=False)

session_name = data['session_names'][Conf.session_idx]
spikes = data['spikes'][Conf.session_idx]
position_vars = data['position_vars'][Conf.session_idx]
position_var_names = data['position_var_names']
task_vars = data['task_vars'][Conf.session_idx]
task_var_names = data['task_var_names']
unit_names = data['unit_names'][Conf.session_idx]
channel_names = data['channel_names'][Conf.session_idx]
unit_types = data['unit_types'][Conf.session_idx]
bin_size = data['bin_size']
bin_times = data['bin_times']
variable_names = np.concatenate((position_var_names, task_var_names), axis=0)

spikes  = np.transpose(spikes, (1, 0, 2)) * 0.2
position_vars  = np.transpose(position_vars, (1, 0, 2))
task_vars = np.transpose(task_vars, (1, 0))

start_idx = np.searchsorted(bin_times, Conf.data.start_time, side='left')
end_idx = np.searchsorted(bin_times, Conf.data.end_time, side='right')

spikes = spikes[:, :, start_idx:end_idx]
position_vars = position_vars[:, :, start_idx:end_idx]
bin_times = bin_times[start_idx:end_idx]

dense = [0, 1, 2]
sparse = [3, 4, 5]

Conf.data.bin_size = bin_size
Conf.data.n_bins = len(bin_times)
Conf.data.n_position_vars = position_vars.shape[1]
Conf.data.n_dense_vars = len(dense)
Conf.data.n_sparse_vars = len(sparse)
Conf.data.n_vars = position_vars.shape[1] + len(dense) + len(sparse)

In [78]:
units_to_remove = [14, 15, 82]
units_mask_delete = np.isin(unit_names, units_to_remove)
units_to_remove_indices = np.where(units_mask_delete)[0]

for unit_idx in units_to_remove_indices:
    plot_unit_spikes(
        spikes=spikes,
        unit_idx=unit_idx,
        title=f"Unit {unit_names[unit_idx]}",
        file_name=f"spike-activity-unit-{unit_idx}",
        file_path=f"./plot/spikes/units/",
        bin_times=bin_times,
        bin_size=Conf.data.bin_size,
    )

unit_names = unit_names[~units_mask_delete]
spikes = spikes[:, ~units_mask_delete]

spike_mask_finite = np.isfinite(spikes).all(axis=(1, 2))
position_vars_mask_finite = np.isfinite(position_vars).all(axis=(1, 2))
task_vars_mask_finite = np.isfinite(task_vars).all(axis=1)
mask_finite = spike_mask_finite & position_vars_mask_finite & task_vars_mask_finite

position_vars_mask_zero = (position_vars == 0).all(axis=(1, 2))

task_vars_mask_interval = np.zeros(task_vars.shape[0], dtype=bool)

for task_var_name in ['tunp', 'tslp']:
    task_var_idx = np.where(task_var_names == task_var_name)[0][0]

    task_var = task_vars[:, task_var_idx]

    task_var_mask_interval = (
        ~np.isfinite(task_var)
        | (task_var <= 2)
        | (task_var >= 60)
    )

    task_vars_mask_interval |= task_var_mask_interval

trials_mask_keep = (
    mask_finite
    & ~position_vars_mask_zero
    & ~task_vars_mask_interval
)

spikes = spikes[trials_mask_keep]
position_vars = position_vars[trials_mask_keep]
task_vars = task_vars[trials_mask_keep]

trials_to_remove, units_to_remove = plot_spike_summary(
    spikes=spikes,
    file_name="spike-summary-before",
    file_path="./plot/spikes",
    metric="outlier_rate",
    k=5.0,
    unit_threshold=0.02,
    trial_threshold=0.02,
    unit_names=unit_names,
    trial_names=np.arange(spikes.shape[0]),
)

trials_mask_delete = np.zeros(spikes.shape[0], dtype=bool)
units_mask_delete = np.zeros(spikes.shape[1], dtype=bool)

trials_mask_delete[trials_to_remove] = True
units_mask_delete[units_to_remove] = True

unit_names = unit_names[~units_mask_delete]
spikes = spikes[~trials_mask_delete][:, ~units_mask_delete, :]

position_vars = position_vars[~trials_mask_delete]
task_vars = task_vars[~trials_mask_delete]

trials_to_remove, units_to_remove = plot_spike_summary(
    spikes=spikes,
    file_name="spike-summary-after",
    file_path="./plot/spikes",
    metric="outlier_rate",
    k=5.0,
    unit_threshold=0.02,
    trial_threshold=0.02,
    unit_names=unit_names,
    trial_names=np.arange(spikes.shape[0]),
)

for task_var_name in ['tunp', 'tslp']:
    task_var_idx = np.where(task_var_names == task_var_name)[0][0]
    task_vars[:, task_var_idx] = np.log(task_vars[:, task_var_idx])

n_trials, n_units , n_bins = spikes.shape
n_trials, n_position_vars, n_bins = position_vars.shape
n_trials, n_task_vars = task_vars.shape

Conf.data.n_units = n_units
Conf.data.n_trials = n_trials

unit_indices = [21]
for unit_idx in unit_indices:
    plot_unit_spikes(
        spikes=spikes,
        unit_idx=unit_idx,
        title=f"Unit {unit_names[unit_idx]}",
        file_name=f"spike-activity-unit-{unit_idx}",
        file_path=f"./plot/spikes/units/",
        bin_times=bin_times,
        bin_size=Conf.data.bin_size,
    )

In [79]:
train_dataset, valid_dataset, Y_mean = prepare_dataset(Conf, position_vars, task_vars[:, dense], task_vars[:, sparse], spikes)
train_loader, valid_loader, loader_generators = prepare_loader(Conf, train_dataset, valid_dataset)

## Model Training

### Compute Results

In [25]:
baseline_trainer, baseline_lit_model = build_lit_model(
    Conf, 
    loader_generators, 
    "baseline", 
    "identity", 
    True
)

baseline_trainer.fit(baseline_lit_model, train_loader, valid_loader)

Output()

In [26]:
mean_trainer, mean_lit_model = build_lit_model(
    Conf,
    loader_generators,
    "conditional",
    "identity",
    True,
)

mean_trainer.fit(mean_lit_model, train_loader, valid_loader)

Output()

In [27]:
mean_cov_trainer, mean_cov_lit_model = build_lit_model(
    Conf,
    loader_generators,
    "conditional",
    "shared",
    True,
)

mean_lit_model_state_dict = mean_lit_model.full_model.mean_model.state_dict()

mean_cov_lit_model.full_model.mean_model.load_state_dict(
    mean_lit_model_state_dict,
    strict=True,
)

for p in mean_cov_lit_model.full_model.mean_model.parameters():
    p.requires_grad = False

mean_cov_lit_model_pre = copy.deepcopy(mean_cov_lit_model)

mean_cov_trainer.fit(mean_cov_lit_model, train_loader, valid_loader)

Output()

### Save Results to Cache

In [28]:
save_cache(
    baseline_lit_model_state=baseline_lit_model.state_dict(),
    mean_lit_model_state=mean_lit_model.state_dict(),
    mean_cov_lit_model_state=mean_cov_lit_model.state_dict(),
    mean_cov_lit_model_pre_state=mean_cov_lit_model_pre.state_dict(),
    baseline_history=baseline_trainer.history,
    mean_history=mean_trainer.history,
    mean_cov_history=mean_cov_trainer.history,
)

Saved: baseline_lit_model_state
Saved: mean_lit_model_state
Saved: mean_cov_lit_model_state
Saved: mean_cov_lit_model_pre_state
Saved: baseline_history
Saved: mean_history
Saved: mean_cov_history


### Load Cached Results

In [29]:
baseline_lit_model_state = load_cache("baseline_lit_model_state")
mean_lit_model_state = load_cache("mean_lit_model_state")
mean_cov_lit_model_state = load_cache("mean_cov_lit_model_state")
mean_cov_lit_model_pre_state = load_cache("mean_cov_lit_model_pre_state")
baseline_history = load_cache("baseline_history")
mean_history = load_cache("mean_history")
mean_cov_history = load_cache("mean_cov_history")

baseline_trainer, baseline_lit_model = build_lit_model(
    Conf,
    loader_generators,
    "baseline",
    "identity",
    True,
)
baseline_lit_model.load_state_dict(baseline_lit_model_state)
baseline_trainer.history = baseline_history

mean_trainer, mean_lit_model = build_lit_model(
    Conf,
    loader_generators,
    "conditional",
    "identity",
    True,
)
mean_lit_model.load_state_dict(mean_lit_model_state)
mean_trainer.history = mean_history

mean_cov_trainer, mean_cov_lit_model = build_lit_model(
    Conf,
    loader_generators,
    "conditional",
    "shared",
    True,
)
mean_cov_lit_model.load_state_dict(mean_cov_lit_model_state)
mean_cov_trainer.history = mean_cov_history

_, mean_cov_lit_model_pre = build_lit_model(
    Conf,
    loader_generators,
    "conditional",
    "shared",
    True,
)
mean_cov_lit_model_pre.load_state_dict(mean_cov_lit_model_pre_state)

Loaded: baseline_lit_model_state
Loaded: mean_lit_model_state
Loaded: mean_cov_lit_model_state
Loaded: mean_cov_lit_model_pre_state
Loaded: baseline_history
Loaded: mean_history
Loaded: mean_cov_history


<All keys matched successfully>

### Visualize and Report Results

In [83]:
plot_training_history(
    trainer=baseline_trainer,
    title="Baseline Model",
    file_name="training-history",
    file_path="./plot/model/baseline/",
)

plot_training_history(
    trainer=mean_trainer,
    title="Mean Model",
    file_name="training-history",
    file_path="./plot/model/mean/",
)

plot_training_history(
    trainer=mean_cov_trainer,
    title="Mean-Cov Model",
    file_name="training-history",
    file_path="./plot/model/mean-cov/",
)

WindowsPath('plot/model/mean-cov')

In [108]:
loading_vmax=max(
    torch.cat([
        x.reshape(-1)
        for x in get_cov_loading_matrices(
            mean_cov_lit_model_pre.full_model.cov_model
        ).values()
    ]).abs().max().item(),
    torch.cat([
        x.reshape(-1)
        for x in get_cov_loading_matrices(
            mean_cov_lit_model.full_model.cov_model
        ).values()
    ]).abs().max().item(),
)

noise_vmax=max(
    torch.sigmoid(
        mean_cov_lit_model_pre.full_model.cov_model.noise
    ).max().item(),
    torch.sigmoid(
        mean_cov_lit_model.full_model.cov_model.noise
    ).max().item(),
)

length_scales_ymax=max(
    torch.cat([
        x.reshape(-1)
        for x in get_cov_length_scales(
            mean_cov_lit_model_pre.full_model.cov_model
        ).values()
    ]).max().item(),
    torch.cat([
        x.reshape(-1)
        for x in get_cov_length_scales(
            mean_cov_lit_model.full_model.cov_model
        ).values()
    ]).max().item(),
)

covariance_vmax=max(
    get_covariance_matrix(
        mean_cov_lit_model_pre.full_model.cov_model
    ).abs().max().item(),
    get_covariance_matrix(
        mean_cov_lit_model.full_model.cov_model
    ).abs().max().item(),
)

unit_indices=[21]

unit_covariance_vmax=torch.stack([
    get_covariance_matrix(
        model.full_model.cov_model
    ).reshape(
        len(bin_times),
        len(unit_names),
        len(bin_times),
        len(unit_names),
    )[
        torch.arange(len(bin_times)),
        unit_idx,
        torch.arange(len(bin_times)),
        :,
    ][
        :,
        torch.arange(len(unit_names))!=unit_idx,
    ].abs().max()
    for model in [
        mean_cov_lit_model_pre,
        mean_cov_lit_model,
    ]
    for unit_idx in unit_indices
]).max().item()

unit_covariance_ymax=torch.stack([
    get_covariance_matrix(
        model.full_model.cov_model
    ).reshape(
        len(bin_times),
        len(unit_names),
        len(bin_times),
        len(unit_names),
    )[
        torch.arange(len(bin_times)),
        unit_idx,
        torch.arange(len(bin_times)),
        :,
    ][
        :,
        torch.arange(len(unit_names))!=unit_idx,
    ].mean(dim=-1).abs().max()
    for model in [
        mean_cov_lit_model_pre,
        mean_cov_lit_model,
    ]
    for unit_idx in unit_indices
]).max().item()

plot_cov_loading_matrix(
    mean_cov_lit_model=mean_cov_lit_model_pre,
    title="Latent Covariance Loadings Before Training",
    file_name="latent-covariance-loadings-pre",
    file_path="./plot/model/mean-cov/parameter/",
    bin_times=bin_times,
    vmax=loading_vmax,
)

plot_cov_noise(
    mean_cov_lit_model=mean_cov_lit_model_pre,
    title="Noise Variance Before Training",
    file_name="noise-variance-pre",
    file_path="./plot/model/mean-cov/parameter/",
    bin_times=bin_times,
    unit_names=unit_names,
    vmax=noise_vmax,
)

plot_cov_length_scales(
    mean_cov_lit_model=mean_cov_lit_model_pre,
    title="Latent Length Scales Before Training",
    file_name="latent-length-scales-pre",
    file_path="./plot/model/mean-cov/parameter/",
    ymax=length_scales_ymax,
)

for time_indices,time_name in zip(
    [None,[13]],
    ["all","13"],
):
    plot_covariance_matrix(
        mean_cov_lit_model=mean_cov_lit_model_pre,
        title="Covariance Matrix Before Training",
        file_name=f"covariance-matrix-pre-{time_name}",
        file_path="./plot/model/mean-cov/parameter/",
        time_indices=time_indices,
        bin_times=bin_times,
        vmax=covariance_vmax,
    )

for unit_idx in unit_indices:
    plot_unit_covariance(
        mean_cov_lit_model=mean_cov_lit_model_pre,
        unit_idx=unit_idx,
        title=f"Unit {unit_names[unit_idx]} Covariance Before Training",
        file_name=f"unit-{unit_idx}-covariance-pre",
        file_path="./plot/model/mean-cov/parameter/unit-covariance/",
        bin_times=bin_times,
        unit_names=unit_names,
        vmax=unit_covariance_vmax,
        ymax=unit_covariance_ymax,
    )

plot_cov_loading_matrix(
    mean_cov_lit_model=mean_cov_lit_model,
    title="Latent Covariance Loadings After Training",
    file_name="latent-covariance-loadings",
    file_path="./plot/model/mean-cov/parameter/",
    bin_times=bin_times,
    vmax=loading_vmax,
)

plot_cov_noise(
    mean_cov_lit_model=mean_cov_lit_model,
    title="Noise Variance After Training",
    file_name="noise-variance",
    file_path="./plot/model/mean-cov/parameter/",
    bin_times=bin_times,
    unit_names=unit_names,
    vmax=noise_vmax,
)

plot_cov_length_scales(
    mean_cov_lit_model=mean_cov_lit_model,
    title="Latent Length Scales After Training",
    file_name="latent-length-scales",
    file_path="./plot/model/mean-cov/parameter/",
    ymax=length_scales_ymax,
)

for time_indices, time_name in zip(
    [None, [0], [2], [4], [6], [8], [10], [12], [13], [14], [16], [18], [20]],
    ["all", "0", "2", "4", "6", "8", "10", "12", "13", "14", "16", "18", "20"],
):
    plot_covariance_matrix(
        mean_cov_lit_model=mean_cov_lit_model,
        title="Covariance Matrix After Training",
        file_name=f"covariance-matrix-{time_name}",
        file_path="./plot/model/mean-cov/parameter/",
        time_indices=time_indices,
        bin_times=bin_times,
        vmax=covariance_vmax,
    )

for unit_idx in unit_indices:
    plot_unit_covariance(
        mean_cov_lit_model=mean_cov_lit_model,
        unit_idx=unit_idx,
        title=f"Unit {unit_names[unit_idx]} Covariance After Training",
        file_name=f"unit-{unit_idx}-covariance",
        file_path="./plot/model/mean-cov/parameter/unit-covariance/",
        bin_times=bin_times,
        unit_names=unit_names,
        vmax=unit_covariance_vmax,
        ymax=unit_covariance_ymax,
    )

## Model Interpretation

### Compute Results

In [93]:
Y_train_baseline, Y_train_baseline_hat = predict_loader(Conf, baseline_lit_model, train_loader, Y_mean, variant="mean")
Y_valid_baseline, Y_valid_baseline_hat = predict_loader(Conf, baseline_lit_model, valid_loader, Y_mean, variant="mean")

Y_train_mean, Y_train_mean_hat = predict_loader(Conf, mean_lit_model, train_loader, Y_mean, variant="mean")
Y_valid_mean, Y_valid_mean_hat = predict_loader(Conf, mean_lit_model, valid_loader, Y_mean, variant="mean")

Y_train_mean_cov, Y_train_mean_cov_hat = predict_loader(Conf, mean_cov_lit_model, train_loader, Y_mean, variant="past_and_others")
Y_valid_mean_cov, Y_valid_mean_cov_hat = predict_loader(Conf, mean_cov_lit_model, valid_loader, Y_mean, variant="past_and_others")

In [95]:
correlation_trial_train_baseline, r2_trial_train_baseline, mse_trial_train_baseline = compute_metrics(Conf, Y_train_baseline, Y_train_baseline_hat)
correlation_unit_train_baseline = np.nanmean(correlation_trial_train_baseline, axis=1)
r2_unit_train_baseline = np.nanmean(r2_trial_train_baseline, axis=1)
mse_unit_train_baseline = np.nanmean(mse_trial_train_baseline, axis=1)


correlation_trial_valid_baseline, r2_trial_valid_baseline, mse_trial_valid_baseline = compute_metrics(Conf, Y_valid_baseline, Y_valid_baseline_hat)
correlation_unit_valid_baseline = np.nanmean(correlation_trial_valid_baseline, axis=1)
r2_unit_valid_baseline = np.nanmean(r2_trial_valid_baseline, axis=1)
mse_unit_valid_baseline = np.nanmean(mse_trial_valid_baseline, axis=1)


correlation_trial_train_mean, r2_trial_train_mean, mse_trial_train_mean = compute_metrics(Conf, Y_train_mean, Y_train_mean_hat)
correlation_unit_train_mean = np.nanmean(correlation_trial_train_mean, axis=1)
r2_unit_train_mean = np.nanmean(r2_trial_train_mean, axis=1)
mse_unit_train_mean = np.nanmean(mse_trial_train_mean, axis=1)


correlation_trial_valid_mean, r2_trial_valid_mean, mse_trial_valid_mean = compute_metrics(Conf, Y_valid_mean, Y_valid_mean_hat)
correlation_unit_valid_mean = np.nanmean(correlation_trial_valid_mean, axis=1)
r2_unit_valid_mean = np.nanmean(r2_trial_valid_mean, axis=1)
mse_unit_valid_mean = np.nanmean(mse_trial_valid_mean, axis=1)


correlation_trial_train_mean_cov, r2_trial_train_mean_cov, mse_trial_train_mean_cov = compute_metrics(Conf, Y_train_mean_cov, Y_train_mean_cov_hat)
correlation_unit_train_mean_cov = np.nanmean(correlation_trial_train_mean_cov, axis=1)
r2_unit_train_mean_cov = np.nanmean(r2_trial_train_mean_cov, axis=1)
mse_unit_train_mean_cov = np.nanmean(mse_trial_train_mean_cov, axis=1)


correlation_trial_valid_mean_cov, r2_trial_valid_mean_cov, mse_trial_valid_mean_cov = compute_metrics(Conf, Y_valid_mean_cov, Y_valid_mean_cov_hat)
correlation_unit_valid_mean_cov = np.nanmean(correlation_trial_valid_mean_cov, axis=1)
r2_unit_valid_mean_cov = np.nanmean(r2_trial_valid_mean_cov, axis=1)
mse_unit_valid_mean_cov = np.nanmean(mse_trial_valid_mean_cov, axis=1)

### Save Results to Cache

In [96]:
save_cache(
    Y_train_baseline=Y_train_baseline,
    Y_train_baseline_hat=Y_train_baseline_hat,
    Y_valid_baseline=Y_valid_baseline,
    Y_valid_baseline_hat=Y_valid_baseline_hat,
    Y_train_mean=Y_train_mean,
    Y_train_mean_hat=Y_train_mean_hat,
    Y_valid_mean=Y_valid_mean,
    Y_valid_mean_hat=Y_valid_mean_hat,
    Y_train_mean_cov=Y_train_mean_cov,
    Y_train_mean_cov_hat=Y_train_mean_cov_hat,
    Y_valid_mean_cov=Y_valid_mean_cov,
    Y_valid_mean_cov_hat=Y_valid_mean_cov_hat,
    correlation_trial_train_baseline=correlation_trial_train_baseline,
    r2_trial_train_baseline=r2_trial_train_baseline,
    mse_trial_train_baseline=mse_trial_train_baseline,
    correlation_unit_train_baseline=correlation_unit_train_baseline,
    r2_unit_train_baseline=r2_unit_train_baseline,
    mse_unit_train_baseline=mse_unit_train_baseline,
    correlation_trial_valid_baseline=correlation_trial_valid_baseline,
    r2_trial_valid_baseline=r2_trial_valid_baseline,
    mse_trial_valid_baseline=mse_trial_valid_baseline,
    correlation_unit_valid_baseline=correlation_unit_valid_baseline,
    r2_unit_valid_baseline=r2_unit_valid_baseline,
    mse_unit_valid_baseline=mse_unit_valid_baseline,
    correlation_trial_train_mean=correlation_trial_train_mean,
    r2_trial_train_mean=r2_trial_train_mean,
    mse_trial_train_mean=mse_trial_train_mean,
    correlation_unit_train_mean=correlation_unit_train_mean,
    r2_unit_train_mean=r2_unit_train_mean,
    mse_unit_train_mean=mse_unit_train_mean,
    correlation_trial_valid_mean=correlation_trial_valid_mean,
    r2_trial_valid_mean=r2_trial_valid_mean,
    mse_trial_valid_mean=mse_trial_valid_mean,
    correlation_unit_valid_mean=correlation_unit_valid_mean,
    r2_unit_valid_mean=r2_unit_valid_mean,
    mse_unit_valid_mean=mse_unit_valid_mean,
    correlation_trial_train_mean_cov=correlation_trial_train_mean_cov,
    r2_trial_train_mean_cov=r2_trial_train_mean_cov,
    mse_trial_train_mean_cov=mse_trial_train_mean_cov,
    correlation_unit_train_mean_cov=correlation_unit_train_mean_cov,
    r2_unit_train_mean_cov=r2_unit_train_mean_cov,
    mse_unit_train_mean_cov=mse_unit_train_mean_cov,
    correlation_trial_valid_mean_cov=correlation_trial_valid_mean_cov,
    r2_trial_valid_mean_cov=r2_trial_valid_mean_cov,
    mse_trial_valid_mean_cov=mse_trial_valid_mean_cov,
    correlation_unit_valid_mean_cov=correlation_unit_valid_mean_cov,
    r2_unit_valid_mean_cov=r2_unit_valid_mean_cov,
    mse_unit_valid_mean_cov=mse_unit_valid_mean_cov,
)

Saved: Y_train_baseline
Saved: Y_train_baseline_hat
Saved: Y_valid_baseline
Saved: Y_valid_baseline_hat
Saved: Y_train_mean
Saved: Y_train_mean_hat
Saved: Y_valid_mean
Saved: Y_valid_mean_hat
Saved: Y_train_mean_cov
Saved: Y_train_mean_cov_hat
Saved: Y_valid_mean_cov
Saved: Y_valid_mean_cov_hat
Saved: correlation_trial_train_baseline
Saved: r2_trial_train_baseline
Saved: mse_trial_train_baseline
Saved: correlation_unit_train_baseline
Saved: r2_unit_train_baseline
Saved: mse_unit_train_baseline
Saved: correlation_trial_valid_baseline
Saved: r2_trial_valid_baseline
Saved: mse_trial_valid_baseline
Saved: correlation_unit_valid_baseline
Saved: r2_unit_valid_baseline
Saved: mse_unit_valid_baseline
Saved: correlation_trial_train_mean
Saved: r2_trial_train_mean
Saved: mse_trial_train_mean
Saved: correlation_unit_train_mean
Saved: r2_unit_train_mean
Saved: mse_unit_train_mean
Saved: correlation_trial_valid_mean
Saved: r2_trial_valid_mean
Saved: mse_trial_valid_mean
Saved: correlation_unit_vali

### Load Cached Results

In [97]:
Y_train_baseline = load_cache("Y_train_baseline")
Y_train_baseline_hat = load_cache("Y_train_baseline_hat")
Y_valid_baseline = load_cache("Y_valid_baseline")
Y_valid_baseline_hat = load_cache("Y_valid_baseline_hat")
Y_train_mean = load_cache("Y_train_mean")
Y_train_mean_hat = load_cache("Y_train_mean_hat")
Y_valid_mean = load_cache("Y_valid_mean")
Y_valid_mean_hat = load_cache("Y_valid_mean_hat")
Y_train_mean_cov = load_cache("Y_train_mean_cov")
Y_train_mean_cov_hat = load_cache("Y_train_mean_cov_hat")
Y_valid_mean_cov = load_cache("Y_valid_mean_cov")
Y_valid_mean_cov_hat = load_cache("Y_valid_mean_cov_hat")

correlation_trial_train_baseline = load_cache("correlation_trial_train_baseline")
r2_trial_train_baseline = load_cache("r2_trial_train_baseline")
mse_trial_train_baseline = load_cache("mse_trial_train_baseline")
correlation_unit_train_baseline = load_cache("correlation_unit_train_baseline")
r2_unit_train_baseline = load_cache("r2_unit_train_baseline")
mse_unit_train_baseline = load_cache("mse_unit_train_baseline")

correlation_trial_valid_baseline = load_cache("correlation_trial_valid_baseline")
r2_trial_valid_baseline = load_cache("r2_trial_valid_baseline")
mse_trial_valid_baseline = load_cache("mse_trial_valid_baseline")
correlation_unit_valid_baseline = load_cache("correlation_unit_valid_baseline")
r2_unit_valid_baseline = load_cache("r2_unit_valid_baseline")
mse_unit_valid_baseline = load_cache("mse_unit_valid_baseline")

correlation_trial_train_mean = load_cache("correlation_trial_train_mean")
r2_trial_train_mean = load_cache("r2_trial_train_mean")
mse_trial_train_mean = load_cache("mse_trial_train_mean")
correlation_unit_train_mean = load_cache("correlation_unit_train_mean")
r2_unit_train_mean = load_cache("r2_unit_train_mean")
mse_unit_train_mean = load_cache("mse_unit_train_mean")

correlation_trial_valid_mean = load_cache("correlation_trial_valid_mean")
r2_trial_valid_mean = load_cache("r2_trial_valid_mean")
mse_trial_valid_mean = load_cache("mse_trial_valid_mean")
correlation_unit_valid_mean = load_cache("correlation_unit_valid_mean")
r2_unit_valid_mean = load_cache("r2_unit_valid_mean")
mse_unit_valid_mean = load_cache("mse_unit_valid_mean")

correlation_trial_train_mean_cov = load_cache("correlation_trial_train_mean_cov")
r2_trial_train_mean_cov = load_cache("r2_trial_train_mean_cov")
mse_trial_train_mean_cov = load_cache("mse_trial_train_mean_cov")
correlation_unit_train_mean_cov = load_cache("correlation_unit_train_mean_cov")
r2_unit_train_mean_cov = load_cache("r2_unit_train_mean_cov")
mse_unit_train_mean_cov = load_cache("mse_unit_train_mean_cov")

correlation_trial_valid_mean_cov = load_cache("correlation_trial_valid_mean_cov")
r2_trial_valid_mean_cov = load_cache("r2_trial_valid_mean_cov")
mse_trial_valid_mean_cov = load_cache("mse_trial_valid_mean_cov")
correlation_unit_valid_mean_cov = load_cache("correlation_unit_valid_mean_cov")
r2_unit_valid_mean_cov = load_cache("r2_unit_valid_mean_cov")
mse_unit_valid_mean_cov = load_cache("mse_unit_valid_mean_cov")

Loaded: Y_train_baseline
Loaded: Y_train_baseline_hat
Loaded: Y_valid_baseline
Loaded: Y_valid_baseline_hat
Loaded: Y_train_mean
Loaded: Y_train_mean_hat
Loaded: Y_valid_mean
Loaded: Y_valid_mean_hat
Loaded: Y_train_mean_cov
Loaded: Y_train_mean_cov_hat
Loaded: Y_valid_mean_cov
Loaded: Y_valid_mean_cov_hat
Loaded: correlation_trial_train_baseline
Loaded: r2_trial_train_baseline
Loaded: mse_trial_train_baseline
Loaded: correlation_unit_train_baseline
Loaded: r2_unit_train_baseline
Loaded: mse_unit_train_baseline
Loaded: correlation_trial_valid_baseline
Loaded: r2_trial_valid_baseline
Loaded: mse_trial_valid_baseline
Loaded: correlation_unit_valid_baseline
Loaded: r2_unit_valid_baseline
Loaded: mse_unit_valid_baseline
Loaded: correlation_trial_train_mean
Loaded: r2_trial_train_mean
Loaded: mse_trial_train_mean
Loaded: correlation_unit_train_mean
Loaded: r2_unit_train_mean
Loaded: mse_unit_train_mean
Loaded: correlation_trial_valid_mean
Loaded: r2_trial_valid_mean
Loaded: mse_trial_valid_

### Visualize and Report Results

In [115]:
unit_indices = [21]
trial_indices = [47]

for unit_idx in unit_indices:
    for trial_idx in trial_indices:
        plot_prediction(
            Y=Y_valid_baseline,
            Y_hat=Y_valid_baseline_hat,
            trial_idx=trial_idx,
            unit_idx=unit_idx,
            bin_times=bin_times,
            title=f"Baseline Model Unit {unit_names[unit_idx]} Prediction",
            filename=f"trial-{trial_idx:03d}-unit-{unit_idx:03d}",
            filepath="./plot/model/baseline/prediction",
        )

        plot_prediction(
            Y=Y_valid_mean,
            Y_hat=Y_valid_mean_hat,
            trial_idx=trial_idx,
            unit_idx=unit_idx,
            bin_times=bin_times,
            title=f"Mean Model Unit {unit_names[unit_idx]} Prediction",
            filename=f"trial-{trial_idx:03d}-unit-{unit_idx:03d}",
            filepath="./plot/model/mean/prediction",
        )

        plot_prediction(
            Y=Y_valid_mean_cov,
            Y_hat=Y_valid_mean_cov_hat,
            trial_idx=trial_idx,
            unit_idx=unit_idx,
            bin_times=bin_times,
            title=f"Mean-Cov Model Unit {unit_names[unit_idx]} Prediction",
            filename=f"trial-{trial_idx:03d}-unit-{unit_idx:03d}",
            filepath="./plot/model/mean-cov/prediction",
        )

In [99]:
plot_train_valid_metrics_comparison(
    correlations_train=correlation_unit_train_baseline,
    correlations_valid=correlation_unit_valid_baseline,
    r2s_train=r2_unit_train_baseline,
    r2s_valid=r2_unit_valid_baseline,
    mses_train=mse_unit_train_baseline,
    mses_valid=mse_unit_valid_baseline,
    title="Baseline Model",
    filename="metrics",
    filepath="./plot/model/baseline/",
)

plot_train_valid_metrics_comparison(
    correlations_train=correlation_unit_train_mean,
    correlations_valid=correlation_unit_valid_mean,
    r2s_train=r2_unit_train_mean,
    r2s_valid=r2_unit_valid_mean,
    mses_train=mse_unit_train_mean,
    mses_valid=mse_unit_valid_mean,
    title="Mean Model",
    filename="metrics",
    filepath="./plot/model/mean/",
)

plot_train_valid_metrics_comparison(
    correlations_train=correlation_unit_train_mean_cov,
    correlations_valid=correlation_unit_valid_mean_cov,
    r2s_train=r2_unit_train_mean_cov,
    r2s_valid=r2_unit_valid_mean_cov,
    mses_train=mse_unit_train_mean_cov,
    mses_valid=mse_unit_valid_mean_cov,
    title="Mean-Covariance Model",
    filename="metrics",
    filepath="./plot/model/mean-cov/",
)

WindowsPath('plot/model/mean-cov')

In [113]:
plot_model_metrics_comparison(
    correlations_x_train=correlation_unit_train_baseline,
    correlations_x_valid=correlation_unit_valid_baseline,
    r2s_x_train=r2_unit_train_baseline,
    r2s_x_valid=r2_unit_valid_baseline,
    mses_x_train=mse_unit_train_baseline,
    mses_x_valid=mse_unit_valid_baseline,
    correlations_y_train=correlation_unit_train_mean,
    correlations_y_valid=correlation_unit_valid_mean,
    r2s_y_train=r2_unit_train_mean,
    r2s_y_valid=r2_unit_valid_mean,
    mses_y_train=mse_unit_train_mean,
    mses_y_valid=mse_unit_valid_mean,
    x_label="Baseline model",
    y_label="Mean model",
    file_name="baseline-vs-mean-metrics",
    file_path="./plot/model/comparison/",
)

plot_model_metrics_comparison(
    correlations_x_train=correlation_unit_train_mean,
    correlations_x_valid=correlation_unit_valid_mean,
    r2s_x_train=r2_unit_train_mean,
    r2s_x_valid=r2_unit_valid_mean,
    mses_x_train=mse_unit_train_mean,
    mses_x_valid=mse_unit_valid_mean,
    correlations_y_train=correlation_unit_train_mean_cov,
    correlations_y_valid=correlation_unit_valid_mean_cov,
    r2s_y_train=r2_unit_train_mean_cov,
    r2s_y_valid=r2_unit_valid_mean_cov,
    mses_y_train=mse_unit_train_mean_cov,
    mses_y_valid=mse_unit_valid_mean_cov,
    x_label="Mean model",
    y_label="Mean-Cov model",
    file_name="mean-vs-mean-cov-metrics",
    file_path="./plot/model/comparison/",
)

WindowsPath('plot/model/comparison')

In [114]:
plot_model_metric_improvement(
    mse_trial_baseline=mse_trial_valid_baseline, 
    mse_trial_model=mse_trial_valid_mean,
    corr_trial_baseline=correlation_trial_valid_baseline, 
    corr_trial_model=correlation_trial_valid_mean,
    r2_trial_baseline=r2_trial_valid_baseline, 
    r2_trial_model=r2_trial_valid_mean,
    title="Baseline vs Mean Model",
    file_name="baseline-vs-mean-improvement",
    file_path="./plot/model/comparison/",
)

plot_model_metric_improvement(
    mse_trial_baseline=mse_trial_valid_mean,
    mse_trial_model=mse_trial_valid_mean_cov,
    corr_trial_baseline=correlation_trial_valid_mean,
    corr_trial_model=correlation_trial_valid_mean_cov,
    r2_trial_baseline=r2_trial_valid_mean,
    r2_trial_model=r2_trial_valid_mean_cov,
    title="Mean vs Mean-Covariance Model",
    file_name="mean-vs-model-improvement",
    file_path="./plot/model/comparison/",
)

WindowsPath('plot/model/comparison')

## Explaining models

### Single-Run SHAP Analysis

#### Compute Results

In [116]:
background, explain, shap_values, base_values = compute_shap_values(Conf, mean_lit_model, train_dataset, valid_dataset)

#### Save Results to Cache

In [117]:
save_cache(
    background=background,
    explain=explain,
    shap_values=shap_values,
    base_values=base_values,
)

Saved: background
Saved: explain
Saved: shap_values
Saved: base_values


#### Load Cached Results

In [118]:
background = load_cache("background")
explain = load_cache("explain")
shap_values = load_cache("shap_values")
base_values = load_cache("base_values")

Loaded: background
Loaded: explain
Loaded: shap_values
Loaded: base_values


#### Visualize and Report Results

In [158]:
unit_bin_indices = [(21, 13)]
trial_indices = [47]

for unit_idx, bin_idx in unit_bin_indices:
    shap_values_selected = shap_values[:, :, bin_idx, unit_idx]
    base_values_selected = base_values[:, bin_idx, unit_idx]

    data = np.concatenate([explain[0][:, bin_idx], explain[1], (explain[2] > 0).astype(int)], axis=1)

    explanation = shap.Explanation(values=shap_values_selected, base_values=base_values_selected, data=data, feature_names=variable_names)

    file_path = f"./plot/model/mean/shap/unit-bin/unit-{unit_idx:03d}-bin-{bin_idx:03d}/"

    for trial_idx in trial_indices:
        save_shap_plot(
            lambda: shap.plots.waterfall(explanation[trial_idx], max_display=len(variable_names), show=False),
            title=f"Unit {unit_names[unit_idx]}, bin {bin_times[bin_idx]} s SHAP attribution",
            file_name=f"trial-{trial_idx:03d}",
            file_path=f"{file_path}waterfall/",
        )

    for variable_name in variable_names:
        save_shap_plot(
            lambda: shap.plots.scatter(explanation[:, variable_name], show=False), 
            title=f"Variable {variable_name}", 
            file_name=f"variable-{variable_name}", 
            file_path=f"{file_path}scatter/",
        )

    save_shap_plot(
        lambda: shap.plots.beeswarm(explanation, max_display=len(variable_names), show=False), 
        title=f"Unit {unit_names[unit_idx]}, bin {bin_times[bin_idx]} s distribution of SHAP attributions", 
        file_name=f"beeswarm", 
        file_path=f"{file_path}summary/",
    )

    save_shap_plot(
        lambda: shap.plots.bar(explanation, max_display=len(variable_names), show=False), 
        title=f"Unit {unit_names[unit_idx]}, bin {bin_times[bin_idx]} s Mean absolute SHAP attribution", 
        file_name=f"bar", 
        file_path=f"{file_path}summary/",
    )

    save_shap_plot(
        lambda: shap.plots.heatmap(explanation, show=False), 
        title=f"Unit {unit_names[unit_idx]}, bin {bin_times[bin_idx]} SHAP attributions", 
        file_name=f"heatmap", 
        file_path=f"{file_path}summary/",
    )

for trial_idx in trial_indices:
    for variable_idx, variable_name in enumerate(variable_names):        
        plot_shap(
            shap_values=shap_values[trial_idx, variable_idx], 
            title=f"Variable {variable_name}", 
            file_name=f"variable-{variable_name}", 
            file_path=f"./plot/model/mean/shap/trial/trial-{trial_idx:03d}/", 
            bin_times=bin_times, 
            unit_names=unit_names,
            cmap=blue_red,
            center_zero=True, 
            show=False,
        )

position, dense, sparse = explain

x = np.concatenate((
    position.transpose(0, 2, 1),
    np.broadcast_to(dense[:, :, None], (*dense.shape, Conf.data.n_bins)),
    np.broadcast_to(sparse[:, :, None], (*sparse.shape, Conf.data.n_bins))
), axis=1)

x = x - x.mean(axis=0, keepdims=True)
s = shap_values - shap_values.mean(axis=0, keepdims=True)

shap_pearson = np.einsum("tvb,tvbu->vbu", x, s) / np.sqrt(
    np.sum(x ** 2, axis=0)[..., None] * np.sum(s ** 2, axis=0)
)

for variable_idx, variable_name in enumerate(variable_names):        
    plot_shap(
        shap_values=shap_pearson[variable_idx], 
        title=f"Variable {variable_name}",
        file_name=f"variable-{variable_name}", 
        file_path=f"./plot/model/mean/shap/population/pearson-shap-input/", 
        bin_times=bin_times, 
        unit_names=unit_names,
        cmap=blue_red,
        center_zero=True,
        show=False,
    )

for variable_idx, variable_name in enumerate(variable_names):        
    plot_shap(
        shap_values=np.abs(shap_values[:, variable_idx]).mean(axis=0), 
        title=f"Variable {variable_name}",
        file_name=f"variable-{variable_name}", 
        file_path=f"./plot/model/mean/shap/population/mean-absolute-shap/", 
        bin_times=bin_times, 
        unit_names=unit_names,
        cmap=white_red, 
        show=False,
    )

### Repeated-Run SHAP Analysis

#### Compute Results

In [ ]:
Conf_shuffle = copy.deepcopy(Conf)
Conf_shuffle.training.patience = 5
Conf_shuffle.optimization.reduce = 2

shap_values_shuffles = []

for shuffle_idx in tqdm(range(Conf.training.n_shuffles), desc="Shuffles"):

    Conf_shuffle.seed = shuffle_idx

    train_dataset_shuffle = shuffle_dataset(Conf_shuffle, train_dataset)
    valid_dataset_shuffle = shuffle_dataset(Conf_shuffle, valid_dataset)
    train_loader_shuffle, valid_loader_shuffle, loader_generators_shuffle = prepare_loader(Conf_shuffle, train_dataset_shuffle, valid_dataset_shuffle)

    mean_trainer_shuffle, mean_lit_model_shuffle = build_lit_model(Conf_shuffle, loader_generators_shuffle, "conditional", "identity", enable_progress_bar_epoch=False)
    
    mean_trainer_shuffle.fit(mean_lit_model_shuffle, train_loader_shuffle, valid_loader_shuffle)
    
    background_shuffle, explain_shuffle, shap_values_shuffle, base_values_shuffle = compute_shap_values(Conf_shuffle, mean_lit_model_shuffle, train_dataset_shuffle, valid_dataset_shuffle)

    shap_values_shuffles.append(shap_values_shuffle)

shap_values_shuffles = np.stack(shap_values_shuffles, axis=0)

In [ ]:
Conf_permutation = copy.deepcopy(Conf)

shap_values_permutations = []

for permutation_idx in tqdm(range(Conf.training.n_permutations), desc="Permutations"):

    Conf_permutation.seed = permutation_idx

    train_loader_permutation, valid_loader_permutation, loader_generators_permutation = prepare_loader(Conf_permutation, train_dataset, valid_dataset)

    mean_trainer_permutation, mean_lit_model_permutation = build_lit_model(Conf_permutation, loader_generators_permutation, "conditional", "identity", enable_progress_bar_epoch=False)
    
    mean_trainer_permutation.fit(mean_lit_model_permutation, train_loader_permutation, valid_loader_permutation)
    
    background_permutation, explain_permutation, shap_values_permutation, base_values_permutation = compute_shap_values(Conf_permutation, mean_lit_model_permutation, train_dataset, valid_dataset)

    shap_values_permutations.append(shap_values_permutation)

shap_values_permutations = np.stack(shap_values_permutations, axis=0)

#### Save Results to Cache

In [ ]:
save_cache(
    shap_values_permutations=shap_values_permutations,
    shap_values_shuffles=shap_values_shuffles,
)

#### Load Cached Results

In [124]:
shap_values_permutations = load_cache("shap_values_permutations")
shap_values_shuffles = load_cache("shap_values_shuffles")

Loaded: shap_values_permutations
Loaded: shap_values_shuffles


#### Visualize and Report Results

In [159]:
shap_values_permutations_mean = np.nanmean(shap_values_permutations, axis=0)

position, dense, sparse = explain

x = np.concatenate((
    position.transpose(0, 2, 1),
    np.broadcast_to(dense[:, :, None], (*dense.shape, Conf.data.n_bins)),
    np.broadcast_to(sparse[:, :, None], (*sparse.shape, Conf.data.n_bins))
), axis=1)

x = x - x.mean(axis=0, keepdims=True)
s = shap_values_permutations_mean - shap_values_permutations_mean.mean(axis=0, keepdims=True)

shap_pearson_permutation = np.einsum("tvb,tvbu->vbu", x, s) / np.sqrt(
    np.sum(x ** 2, axis=0)[..., None] * np.sum(s ** 2, axis=0) + 1e-12
)

for variable_idx, variable_name in enumerate(variable_names):
    plot_shap(
        shap_values=shap_pearson_permutation[variable_idx],
        title=f"Variable-{variable_name}",
        file_name=f"variable-{variable_name}",
        file_path="./plot/model/mean/shap-permutation/population/pearson-shap-input/",
        bin_times=bin_times,
        unit_names=unit_names,
        cmap=blue_red,
        center_zero=True,
        show=False,
    )

for variable_idx, variable_name in enumerate(variable_names):
    plot_shap(
        shap_values=np.abs(shap_values_permutations_mean[:, variable_idx]).mean(axis=0),
        title=f"Variable {variable_name}",
        file_name=f"variable-{variable_name}",
        file_path="./plot/model/mean/shap-permutation/population/mean-absolute-shap/",
        bin_times=bin_times,
        unit_names=unit_names,
        cmap=white_red,
        show=False,
    )

In [163]:
variable_colors = ["black",  "black", "black",  "black",  "blue",  "green", "green", "red", "blue", "green"]

for trial_idx in trial_indices:
    for unit_idx, bin_idx in unit_bin_indices:
        for variable_idx, variable_name in enumerate(variable_names):
            plot_shap_hist(
                unit_bin_shap_shuffles=shap_values_shuffles[:, trial_idx, variable_idx, bin_idx, unit_idx],
                unit_bin_shap_permutations=shap_values_permutations[:, trial_idx, variable_idx, bin_idx, unit_idx],
                variable_name=variable_name,
                unit_name=unit_names[unit_idx],
                bin_time=bin_times[bin_idx],
                color=variable_colors[variable_idx],
                title=f"variable {variable_name}",
                filename=f"variable-{variable_name}",
                filepath=f"./plot/model/mean/shap-shuffle-permutation/unit-bin/unit-{unit_idx:03d}-bin-{bin_idx:03d}/trial-{trial_idx:03d}",
            )

In [161]:
shap_values_permutations_mean = np.nanmean(shap_values_permutations, axis=0)
shap_values_shuffles_mean = np.nanmean(shap_values_shuffles, axis=0)

shap_values_permutations_var = np.nanvar(shap_values_permutations, axis=0, ddof=1)
shap_values_shuffles_var = np.nanvar(shap_values_shuffles, axis=0, ddof=1)

shap_values_var = (
    (Conf.training.n_permutations - 1) * shap_values_permutations_var
    + (Conf.training.n_shuffles - 1) * shap_values_shuffles_var
) / (
    Conf.training.n_permutations + Conf.training.n_shuffles - 2
)

shap_values_standardized_null = (
    shap_values_permutations_mean - shap_values_shuffles_mean
) / np.sqrt(shap_values_var + 1e-12)

position, dense, sparse = explain
x = np.concatenate([
    position.transpose(0, 2, 1),
    np.broadcast_to(dense[:, :, None], (*dense.shape, Conf.data.n_bins)),
    np.broadcast_to(sparse[:, :, None], (*sparse.shape, Conf.data.n_bins)),
], axis=1)

x = x - x.mean(axis=0, keepdims=True)
s = shap_values_standardized_null - shap_values_standardized_null.mean(axis=0, keepdims=True)

shap_pearson_standardized_null = np.einsum("tvb,tvbu->vbu", x, s) / (
    np.sqrt(np.sum(x ** 2, axis=0)[..., None] * np.sum(s ** 2, axis=0)) + 1e-12
)

for variable_idx, variable_name in enumerate(variable_names):
    plot_shap(
        shap_values=shap_pearson_standardized_null[variable_idx],
        title=f"variable-{variable_name}",
        file_name=f"variable-{variable_name}",
        file_path="./plot/model/mean/shap-standardized-null/population/pearson-shap-input",
        bin_times=bin_times,
        unit_names=unit_names,
        cmap=blue_red,
        center_zero=True,
        show=False,
    )

for variable_idx, variable_name in enumerate(variable_names):
    plot_shap(
        shap_values=np.abs(shap_values_standardized_null[:, variable_idx]).mean(axis=0),
        title=f"variable-{variable_name}",
        file_name=f"variable-{variable_name}",
        file_path="./plot/model/mean/shap-standardized-null/population/mean-absolute-shap",
        bin_times=bin_times,
        unit_names=unit_names,
        cmap=white_red,
        show=False,
    )

## Functional Selectivity Analysis

### Compute Results

In [ ]:
variable_time_windows = {
    "x": (-1, 2),
    "y": (-1, 2),
    "d": (-1, 2),
    "motion": (-1, 2),
    "tslp": (-1, 2),
    "last_choice": (-1, 2),
    "rew_ratio": (-1, 2),
    "rew": (-1, 2),
    "tunp": (-1, 2),
    "choice": (-1, 2),
}

variable_selectivity_permutations = np.zeros((Conf.training.n_permutations, len(variable_names), n_units))
variable_selectivity_shuffles = np.zeros((Conf.training.n_shuffles, len(variable_names), n_units))

for variable_idx, variable_name in enumerate(variable_names):
    time_window = variable_time_windows[variable_name]
    time_mask = (bin_times >= time_window[0]) & (bin_times <= time_window[1])

    var_shap_permutations = shap_values_permutations[:, :, variable_idx, time_mask, :]
    var_shap_shuffles = shap_values_shuffles[:, :, variable_idx, time_mask, :]

    variable_selectivity_permutations[:, variable_idx, :] = np.nanmax(np.nanmean(np.abs(var_shap_permutations), axis=(1)), axis=(1))
    variable_selectivity_shuffles[:, variable_idx, :] = np.nanmax(np.nanmean(np.abs(var_shap_shuffles), axis=(1)), axis=(1))

In [ ]:
alpha = 0.05
minimum_standardized_null = 5

variable_selectivity_permutations_mean = np.nanmean(variable_selectivity_permutations, axis=0)
variable_selectivity_shuffles_mean = np.nanmean(variable_selectivity_shuffles, axis=0)

variable_selectivity_permutations_var = np.nanvar(variable_selectivity_permutations, axis=0, ddof=1)
variable_selectivity_shuffles_var = np.nanvar(variable_selectivity_shuffles, axis=0, ddof=1)

variable_selectivity_var = (
    (Conf.training.n_permutations - 1) * variable_selectivity_permutations_var
    + (Conf.training.n_shuffles - 1) * variable_selectivity_shuffles_var
) / (
    Conf.training.n_permutations + Conf.training.n_shuffles - 2
)

variable_selectivity_standardized_null = (
    variable_selectivity_permutations_mean - variable_selectivity_shuffles_mean
) / np.sqrt(variable_selectivity_var + 1e-12)

variable_selectivity_t, variable_selectivity_p = stats.ttest_ind(
    variable_selectivity_permutations,
    variable_selectivity_shuffles,
    axis=0,
    equal_var=False,
    alternative="greater",
    nan_policy="omit",
)

variable_selectivity_q = np.full(variable_selectivity_p.shape, np.nan)
valid_selectivity_p = np.isfinite(variable_selectivity_p)
variable_selectivity_q[valid_selectivity_p] = multipletests(variable_selectivity_p[valid_selectivity_p], alpha=alpha, method="fdr_bh")[1]

variable_selectivity_significant = (variable_selectivity_q < alpha) & (variable_selectivity_standardized_null >= minimum_standardized_null)

neuron_classes = {
    "movement": ["x", "y", "d", "motion"],
    "reward_prediction": ["tslp", "last_choice", "rew_ratio"],
    "reward_outcome": ["rew"],
    "action_planning": ["tunp", "choice"],
}
class_names = list(neuron_classes.keys())

class_selectivity_significant = np.array([
    np.any(variable_selectivity_significant[np.isin(variable_names, neuron_classes[class_name])], axis=0)
    for class_name in class_names
])

movement = class_selectivity_significant[class_names.index("movement")]
reward_prediction = class_selectivity_significant[class_names.index("reward_prediction")]
reward_outcome = class_selectivity_significant[class_names.index("reward_outcome")]
action_planning = class_selectivity_significant[class_names.index("action_planning")]

### Save Results to Cache

In [ ]:
save_cache(
    variable_selectivity_permutations=variable_selectivity_permutations,
    variable_selectivity_shuffles=variable_selectivity_shuffles,
    variable_selectivity_standardized_null=variable_selectivity_standardized_null,
    variable_selectivity_p=variable_selectivity_p,
    variable_selectivity_q=variable_selectivity_q,
    variable_selectivity_significant=variable_selectivity_significant,
    class_selectivity_significant=class_selectivity_significant,
    movement=movement,
    reward_prediction=reward_prediction,
    reward_outcome=reward_outcome,
    action_planning=action_planning,
)

### Load Cached Results

In [164]:
variable_selectivity_permutations = load_cache("variable_selectivity_permutations")
variable_selectivity_shuffles = load_cache("variable_selectivity_shuffles")
variable_selectivity_standardized_null = load_cache("variable_selectivity_standardized_null")
variable_selectivity_p = load_cache("variable_selectivity_p")
variable_selectivity_q = load_cache("variable_selectivity_q")
variable_selectivity_significant = load_cache("variable_selectivity_significant")
class_selectivity_significant = load_cache("class_selectivity_significant")
movement = load_cache("movement")
reward_prediction = load_cache("reward_prediction")
reward_outcome = load_cache("reward_outcome")
action_planning = load_cache("action_planning")

neuron_classes = {
    "movement": ["x", "y", "d", "motion"],
    "reward_prediction": ["tslp", "last_choice", "rew_rate", "rew_ratio"],
    "reward_outcome": ["rew"],
    "action_planning": ["tunp", "choice"],
}

class_names = list(neuron_classes.keys())

Loaded: variable_selectivity_permutations
Loaded: variable_selectivity_shuffles
Loaded: variable_selectivity_standardized_null
Loaded: variable_selectivity_p
Loaded: variable_selectivity_q
Loaded: variable_selectivity_significant
Loaded: class_selectivity_significant
Loaded: movement
Loaded: reward_prediction
Loaded: reward_outcome
Loaded: action_planning


### Visualize and Report Results

In [180]:
class_color_map = {
    "movement": "black",
    "reward_prediction": "green",
    "reward_outcome": "red",
    "action_planning": "blue",
}

variable_to_class = {
    var: cls
    for cls, vars_list in neuron_classes.items()
    for var in vars_list
}
variable_colors = [class_color_map[variable_to_class[var]] for var in variable_names]
class_colors = [class_color_map[cls] for cls in class_names]

variable_selectivity_mean = np.nanmean(variable_selectivity_permutations, axis=0)

unit_indices = [21]

for variable_idx, variable_name in enumerate(variable_names):
    color = variable_colors[variable_idx]
    for unit_idx in unit_indices:
        plot_variable_selectivity_hist(
            unit_selectivity_shuffles=variable_selectivity_shuffles[:, variable_idx, unit_idx],
            unit_selectivity_permutations=variable_selectivity_permutations[:, variable_idx, unit_idx],
            variable_name=variable_name,
            unit_name=unit_names[unit_idx],
            color=color,
            title=f"Variable {variable_name}, Unit {unit_names[unit_idx]}",
            filename=f"variable-{variable_name}-unit-{unit_idx:03d}",
            filepath=f"./plot/selectivity/variable_selectivity/variable_selectivity_hist/",
        )

for variable_idx, variable_name in enumerate(variable_names):
    plot_variable_selectivity_curves(
        variable_selectivity=variable_selectivity_standardized_null[variable_idx],
        variable_name=variable_name,
        significant=variable_selectivity_significant[variable_idx],
        color=variable_colors[variable_idx],
        title=f"Variable {variable_name}",
        filename=f"variable-selectivity-curves-{variable_name}",
        filepath="./plot/selectivity/variable_selectivity/variable_selectivity_curves",
    )

for variable_idx, variable_name in enumerate(variable_names):
    plot_variable_selectivity_pca(
        variable_selectivities=variable_selectivity_standardized_null,
        variable_selectivity=variable_selectivity_mean[variable_idx],
        variable_name=variable_name,
        significant=variable_selectivity_significant[variable_idx],
        color=variable_colors[variable_idx],
        title=f"Variable {variable_name}",
        filename=f"variable-selectivity-pca-{variable_name}",
        filepath="./plot/selectivity/variable_selectivity/variable_selectivity_pca",
    )

for class_idx, class_name in enumerate(class_names):
    plot_class_selectivity_pca(
        variable_selectivity=variable_selectivity_standardized_null,
        class_name=class_name,
        significant=class_selectivity_significant[class_idx],
        color=class_colors[class_idx],
        title=f"Class {class_name.replace('_', ' ')}",
        filename=f"class-selectivity-pca-{class_name}",
        filepath="./plot/selectivity/class_selectivity/class_selectivity_pca",
    )

plot_variable_selectivity_matrix(
    variable_selectivity=variable_selectivity_standardized_null,
    variable_names=variable_names,
    unit_names=unit_names,
    significant=variable_selectivity_significant,
    variable_colors=variable_colors,
    title="Variable Selectivity Matrix",
    filename="variable-selectivity-matrix",
    filepath="./plot/selectivity/variable_selectivity",
)

class_selectivity_standardized_null = np.array([
    np.nanmean(variable_selectivity_standardized_null[np.isin(variable_names, neuron_classes[class_name])], axis=0)
    for class_name in class_names
])

plot_class_selectivity_matrix(
    class_selectivity=class_selectivity_standardized_null,
    class_names=class_names,
    unit_names=unit_names,
    significant=class_selectivity_significant,
    class_colors=class_colors,
    title="Class Selectivity Matrix",
    filename="class-selectivity-matrix",
    filepath="./plot/selectivity/class_selectivity",
)

WindowsPath('plot/selectivity/class_selectivity')

In [181]:
print(f"reward_prediction only: {np.sum(reward_prediction & ~reward_outcome & ~action_planning)}")
print(f"reward_outcome only: {np.sum(~reward_prediction & reward_outcome & ~action_planning)}")
print(f"action_planning only: {np.sum(~reward_prediction & ~reward_outcome & action_planning)}")
print(f"reward_prediction + reward_outcome only: {np.sum(reward_prediction & reward_outcome & ~action_planning)}")
print(f"reward_prediction + action_planning only: {np.sum(reward_prediction & ~reward_outcome & action_planning)}")
print(f"reward_outcome + action_planning only: {np.sum(~reward_prediction & reward_outcome & action_planning)}")
print(f"all three: {np.sum(reward_prediction & reward_outcome & action_planning)}")
print(f"none: {np.sum(~reward_prediction & ~reward_outcome & ~action_planning)}")

print("")

print("reward_prediction and reward_outcome")
print(f"reward_prediction only: {np.sum(reward_prediction & ~reward_outcome)}")
print(f"reward_outcome only: {np.sum(~reward_prediction & reward_outcome)}")
print(f"reward_prediction + reward_outcome: {np.sum(reward_prediction & reward_outcome)}")
print(f"none: {np.sum(~reward_prediction & ~reward_outcome)}")

print("")

print("reward_prediction and action_planning")
print(f"reward_prediction only: {np.sum(reward_prediction & ~action_planning)}")
print(f"action_planning only: {np.sum(~reward_prediction & action_planning)}")
print(f"reward_prediction + action_planning: {np.sum(reward_prediction & action_planning)}")
print(f"none: {np.sum(~reward_prediction & ~action_planning)}")

print("")

print("reward_outcome and action_planning")
print(f"reward_outcome only: {np.sum(reward_outcome & ~action_planning)}")
print(f"action_planning only: {np.sum(~reward_outcome & action_planning)}")
print(f"reward_outcome + action_planning: {np.sum(reward_outcome & action_planning)}")
print(f"none: {np.sum(~reward_outcome & ~action_planning)}")

reward_prediction only: 4
reward_outcome only: 15
action_planning only: 3
reward_prediction + reward_outcome only: 15
reward_prediction + action_planning only: 4
reward_outcome + action_planning only: 7
all three: 20
none: 8

reward_prediction and reward_outcome
reward_prediction only: 8
reward_outcome only: 22
reward_prediction + reward_outcome: 35
none: 11

reward_prediction and action_planning
reward_prediction only: 19
action_planning only: 10
reward_prediction + action_planning: 24
none: 23

reward_outcome and action_planning
reward_outcome only: 30
action_planning only: 7
reward_outcome + action_planning: 27
none: 12


## Cross-Group Covariance Analysis

### Compute Results

In [ ]:
Conf_prediction_outcome = copy.deepcopy(Conf)

group_names_prediction_outcome = np.array(
    ["prediction_only", "outcome_only", "both", "neither"],
    dtype=object,
)

unit_group_idx = np.full(n_units, 3, dtype=int)
unit_group_idx[reward_prediction & ~reward_outcome] = 0
unit_group_idx[~reward_prediction & reward_outcome] = 1
unit_group_idx[reward_prediction & reward_outcome] = 2

Conf_prediction_outcome.data.unit_groups = group_names_prediction_outcome[unit_group_idx].tolist()

Conf_prediction_outcome.model_type.cov.components = {
    "prediction_only": {
        "groups": ["prediction_only"],
        "n_latent": 2,
    },
    "outcome_only": {
        "groups": ["outcome_only"],
        "n_latent": 2,
    },
    "both": {
        "groups": ["both"],
        "n_latent": 2,
    },
    "neither": {
        "groups": ["neither"],
        "n_latent": 2,
    },
    "prediction_outcome": {
        "groups": ["prediction_only", "outcome_only"],
        "n_latent": 2,
    },
    "prediction_both": {
        "groups": ["prediction_only", "both"],
        "n_latent": 2,
    },
    "prediction_neither": {
        "groups": ["prediction_only", "neither"],
        "n_latent": 2,
    },
    "outcome_both": {
        "groups": ["outcome_only", "both"],
        "n_latent": 2,
    },
    "outcome_neither": {
        "groups": ["outcome_only", "neither"],
        "n_latent": 2,
    },
    "both_neither": {
        "groups": ["both", "neither"],
        "n_latent": 2,
    },
}

Conf_prediction_outcome.model_type.cov.n_latent = sum(
    config["n_latent"]
    for config in Conf_prediction_outcome.model_type.cov.components.values()
)

prediction_outcome_trainer, prediction_outcome_lit_model = build_lit_model(
    Conf_prediction_outcome,
    loader_generators,
    "conditional",
    "functional_groups",
    True,
)

prediction_outcome_lit_model.full_model.mean_model.load_state_dict(
    mean_lit_model.full_model.mean_model.state_dict(),
    strict=True,
)

for p in prediction_outcome_lit_model.full_model.mean_model.parameters():
    p.requires_grad = False

prediction_outcome_trainer.fit(
    prediction_outcome_lit_model,
    train_loader,
    valid_loader,
)

correlation_matrix_prediction_outcome_over_mean, r2_matrix_prediction_outcome_over_mean, mse_matrix_prediction_outcome_over_mean = (
    compute_group_influence_matrix(
        Conf=Conf_prediction_outcome,
        lit_model=prediction_outcome_lit_model,
        loader=valid_loader,
        Y_mean=Y_mean,
        group_names=group_names_prediction_outcome,
        baseline="mean",
    )
)

correlation_matrix_prediction_outcome_over_within, r2_matrix_prediction_outcome_over_within, mse_matrix_prediction_outcome_over_within = (
    compute_group_influence_matrix(
        Conf=Conf_prediction_outcome,
        lit_model=prediction_outcome_lit_model,
        loader=valid_loader,
        Y_mean=Y_mean,
        group_names=group_names_prediction_outcome,
        baseline="within",
    )
)

In [190]:
Conf_prediction_planning = copy.deepcopy(Conf)

group_names_prediction_planning = np.array(
    ["prediction_only", "planning_only", "both", "neither"],
    dtype=object,
)

unit_group_idx = np.full(n_units, 3, dtype=int)
unit_group_idx[reward_prediction & ~action_planning] = 0
unit_group_idx[~reward_prediction & action_planning] = 1
unit_group_idx[reward_prediction & action_planning] = 2

Conf_prediction_planning.data.unit_groups = group_names_prediction_planning[unit_group_idx].tolist()

Conf_prediction_planning.model_type.cov.components = {
    "prediction_only": {
        "groups": ["prediction_only"],
        "n_latent": 2,
    },
    "planning_only": {
        "groups": ["planning_only"],
        "n_latent": 2,
    },
    "both": {
        "groups": ["both"],
        "n_latent": 2,
    },
    "neither": {
        "groups": ["neither"],
        "n_latent": 2,
    },
    "prediction_planning": {
        "groups": ["prediction_only", "planning_only"],
        "n_latent": 2,
    },
    "prediction_both": {
        "groups": ["prediction_only", "both"],
        "n_latent": 2,
    },
    "prediction_neither": {
        "groups": ["prediction_only", "neither"],
        "n_latent": 2,
    },
    "planning_both": {
        "groups": ["planning_only", "both"],
        "n_latent": 2,
    },
    "planning_neither": {
        "groups": ["planning_only", "neither"],
        "n_latent": 2,
    },
    "both_neither": {
        "groups": ["both", "neither"],
        "n_latent": 2,
    },
}

Conf_prediction_planning.model_type.cov.n_latent = sum(
    config["n_latent"]
    for config in Conf_prediction_planning.model_type.cov.components.values()
)

prediction_planning_trainer, prediction_planning_lit_model = build_lit_model(
    Conf_prediction_planning,
    loader_generators,
    "conditional",
    "functional_groups",
    True,
)

prediction_planning_lit_model.full_model.mean_model.load_state_dict(
    mean_lit_model.full_model.mean_model.state_dict(),
    strict=True,
)

for p in prediction_planning_lit_model.full_model.mean_model.parameters():
    p.requires_grad = False

prediction_planning_trainer.fit(
    prediction_planning_lit_model,
    train_loader,
    valid_loader,
)

correlation_matrix_prediction_planning_over_mean, r2_matrix_prediction_planning_over_mean, mse_matrix_prediction_planning_over_mean = (
    compute_group_influence_matrix(
        Conf=Conf_prediction_planning,
        lit_model=prediction_planning_lit_model,
        loader=valid_loader,
        Y_mean=Y_mean,
        group_names=group_names_prediction_planning,
        baseline="mean",
    )
)

correlation_matrix_prediction_planning_over_within, r2_matrix_prediction_planning_over_within, mse_matrix_prediction_planning_over_within = (
    compute_group_influence_matrix(
        Conf=Conf_prediction_planning,
        lit_model=prediction_planning_lit_model,
        loader=valid_loader,
        Y_mean=Y_mean,
        group_names=group_names_prediction_planning,
        baseline="within",
    )
)

Output()

c:\Users\A\miniconda3\lib\site-packages\scipy\stats\_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


In [ ]:
Conf_outcome_planning = copy.deepcopy(Conf)

group_names_outcome_planning = np.array(
    ["outcome_only", "planning_only", "both", "neither"],
    dtype=object,
)

unit_group_idx = np.full(n_units, 3, dtype=int)
unit_group_idx[reward_outcome & ~action_planning] = 0
unit_group_idx[~reward_outcome & action_planning] = 1
unit_group_idx[reward_outcome & action_planning] = 2

Conf_outcome_planning.data.unit_groups = group_names_outcome_planning[unit_group_idx].tolist()

Conf_outcome_planning.model_type.cov.components = {
    "outcome_only": {
        "groups": ["outcome_only"],
        "n_latent": 2,
    },
    "planning_only": {
        "groups": ["planning_only"],
        "n_latent": 2,
    },
    "both": {
        "groups": ["both"],
        "n_latent": 2,
    },
    "neither": {
        "groups": ["neither"],
        "n_latent": 2,
    },
    "outcome_planning": {
        "groups": ["outcome_only", "planning_only"],
        "n_latent": 2,
    },
    "outcome_both": {
        "groups": ["outcome_only", "both"],
        "n_latent": 2,
    },
    "outcome_neither": {
        "groups": ["outcome_only", "neither"],
        "n_latent": 2,
    },
    "planning_both": {
        "groups": ["planning_only", "both"],
        "n_latent": 2,
    },
    "planning_neither": {
        "groups": ["planning_only", "neither"],
        "n_latent": 2,
    },
    "both_neither": {
        "groups": ["both", "neither"],
        "n_latent": 2,
    },
}

Conf_outcome_planning.model_type.cov.n_latent = sum(
    config["n_latent"]
    for config in Conf_outcome_planning.model_type.cov.components.values()
)

outcome_planning_trainer, outcome_planning_lit_model = build_lit_model(
    Conf_outcome_planning,
    loader_generators,
    "conditional",
    "functional_groups",
    True,
)

outcome_planning_lit_model.full_model.mean_model.load_state_dict(
    mean_lit_model.full_model.mean_model.state_dict(),
    strict=True,
)

for p in outcome_planning_lit_model.full_model.mean_model.parameters():
    p.requires_grad = False

outcome_planning_trainer.fit(
    outcome_planning_lit_model,
    train_loader,
    valid_loader,
)

correlation_matrix_outcome_planning_over_mean, r2_matrix_outcome_planning_over_mean, mse_matrix_outcome_planning_over_mean = (
    compute_group_influence_matrix(
        Conf=Conf_outcome_planning,
        lit_model=outcome_planning_lit_model,
        loader=valid_loader,
        Y_mean=Y_mean,
        group_names=group_names_outcome_planning,
        baseline="mean",
    )
)

correlation_matrix_outcome_planning_over_within, r2_matrix_outcome_planning_over_within, mse_matrix_outcome_planning_over_within = (
    compute_group_influence_matrix(
        Conf=Conf_outcome_planning,
        lit_model=outcome_planning_lit_model,
        loader=valid_loader,
        Y_mean=Y_mean,
        group_names=group_names_outcome_planning,
        baseline="within",
    )
)

### Save Results to Cache

In [ ]:
save_cache(
    Conf_prediction_planning=Conf_prediction_planning,
    group_names_prediction_planning=group_names_prediction_planning,
    prediction_planning_lit_model_state=prediction_planning_lit_model.state_dict(),
    prediction_planning_trainer=prediction_planning_trainer,
    correlation_matrix_prediction_planning_over_mean=correlation_matrix_prediction_planning_over_mean,
    r2_matrix_prediction_planning_over_mean=r2_matrix_prediction_planning_over_mean,
    mse_matrix_prediction_planning_over_mean=mse_matrix_prediction_planning_over_mean,
    correlation_matrix_prediction_planning_over_within=correlation_matrix_prediction_planning_over_within,
    r2_matrix_prediction_planning_over_within=r2_matrix_prediction_planning_over_within,
    mse_matrix_prediction_planning_over_within=mse_matrix_prediction_planning_over_within,
    Conf_prediction_outcome=Conf_prediction_outcome,
    group_names_prediction_outcome=group_names_prediction_outcome,
    prediction_outcome_lit_model_state=prediction_outcome_lit_model.state_dict(),
    prediction_outcome_trainer=prediction_outcome_trainer,
    correlation_matrix_prediction_outcome_over_mean=correlation_matrix_prediction_outcome_over_mean,
    r2_matrix_prediction_outcome_over_mean=r2_matrix_prediction_outcome_over_mean,
    mse_matrix_prediction_outcome_over_mean=mse_matrix_prediction_outcome_over_mean,
    correlation_matrix_prediction_outcome_over_within=correlation_matrix_prediction_outcome_over_within,
    r2_matrix_prediction_outcome_over_within=r2_matrix_prediction_outcome_over_within,
    mse_matrix_prediction_outcome_over_within=mse_matrix_prediction_outcome_over_within,
    Conf_outcome_planning=Conf_outcome_planning,
    group_names_outcome_planning=group_names_outcome_planning,
    outcome_planning_lit_model_state=outcome_planning_lit_model.state_dict(),
    outcome_planning_trainer=outcome_planning_trainer,
    correlation_matrix_outcome_planning_over_mean=correlation_matrix_outcome_planning_over_mean,
    r2_matrix_outcome_planning_over_mean=r2_matrix_outcome_planning_over_mean,
    mse_matrix_outcome_planning_over_mean=mse_matrix_outcome_planning_over_mean,
    correlation_matrix_outcome_planning_over_within=correlation_matrix_outcome_planning_over_within,
    r2_matrix_outcome_planning_over_within=r2_matrix_outcome_planning_over_within,
    mse_matrix_outcome_planning_over_within=mse_matrix_outcome_planning_over_within,
)

### Load Cached Results

In [172]:
Conf_prediction_planning = load_cache("Conf_prediction_planning")
group_names_prediction_planning = load_cache("group_names_prediction_planning")
prediction_planning_lit_model_state = load_cache("prediction_planning_lit_model_state")
correlation_matrix_prediction_planning_over_mean = load_cache(
    "correlation_matrix_prediction_planning_over_mean"
)
r2_matrix_prediction_planning_over_mean = load_cache(
    "r2_matrix_prediction_planning_over_mean"
)
mse_matrix_prediction_planning_over_mean = load_cache(
    "mse_matrix_prediction_planning_over_mean"
)
correlation_matrix_prediction_planning_over_within = load_cache(
    "correlation_matrix_prediction_planning_over_within"
)
r2_matrix_prediction_planning_over_within = load_cache(
    "r2_matrix_prediction_planning_over_within"
)
mse_matrix_prediction_planning_over_within = load_cache(
    "mse_matrix_prediction_planning_over_within"
)


Conf_prediction_outcome = load_cache("Conf_prediction_outcome")
group_names_prediction_outcome = load_cache("group_names_prediction_outcome")
prediction_outcome_lit_model_state = load_cache("prediction_outcome_lit_model_state")
correlation_matrix_prediction_outcome_over_mean = load_cache(
    "correlation_matrix_prediction_outcome_over_mean"
)
r2_matrix_prediction_outcome_over_mean = load_cache(
    "r2_matrix_prediction_outcome_over_mean"
)
mse_matrix_prediction_outcome_over_mean = load_cache(
    "mse_matrix_prediction_outcome_over_mean"
)
correlation_matrix_prediction_outcome_over_within = load_cache(
    "correlation_matrix_prediction_outcome_over_within"
)
r2_matrix_prediction_outcome_over_within = load_cache(
    "r2_matrix_prediction_outcome_over_within"
)
mse_matrix_prediction_outcome_over_within = load_cache(
    "mse_matrix_prediction_outcome_over_within"
)


Conf_outcome_planning = load_cache("Conf_outcome_planning")
group_names_outcome_planning = load_cache("group_names_outcome_planning")
outcome_planning_lit_model_state = load_cache("outcome_planning_lit_model_state")
correlation_matrix_outcome_planning_over_mean = load_cache(
    "correlation_matrix_outcome_planning_over_mean"
)
r2_matrix_outcome_planning_over_mean = load_cache(
    "r2_matrix_outcome_planning_over_mean"
)
mse_matrix_outcome_planning_over_mean = load_cache(
    "mse_matrix_outcome_planning_over_mean"
)
correlation_matrix_outcome_planning_over_within = load_cache(
    "correlation_matrix_outcome_planning_over_within"
)
r2_matrix_outcome_planning_over_within = load_cache(
    "r2_matrix_outcome_planning_over_within"
)
mse_matrix_outcome_planning_over_within = load_cache(
    "mse_matrix_outcome_planning_over_within"
)


prediction_planning_trainer, prediction_planning_lit_model = build_lit_model(
    Conf_prediction_planning,
    loader_generators,
    "conditional",
    "functional_groups",
    True,
)
prediction_planning_lit_model.load_state_dict(
    prediction_planning_lit_model_state
)
prediction_planning_lit_model.eval()


prediction_outcome_trainer, prediction_outcome_lit_model = build_lit_model(
    Conf_prediction_outcome,
    loader_generators,
    "conditional",
    "functional_groups",
    True,
)
prediction_outcome_lit_model.load_state_dict(
    prediction_outcome_lit_model_state
)
prediction_outcome_lit_model.eval()


outcome_planning_trainer, outcome_planning_lit_model = build_lit_model(
    Conf_outcome_planning,
    loader_generators,
    "conditional",
    "functional_groups",
    True,
)
outcome_planning_lit_model.load_state_dict(
    outcome_planning_lit_model_state
)
outcome_planning_lit_model.eval()

Loaded: Conf_prediction_planning
Loaded: group_names_prediction_planning
Loaded: prediction_planning_lit_model_state
Variable 'correlation_matrix_prediction_planning_over_mean' was not found in cache.
Variable 'r2_matrix_prediction_planning_over_mean' was not found in cache.
Variable 'mse_matrix_prediction_planning_over_mean' was not found in cache.
Variable 'correlation_matrix_prediction_planning_over_within' was not found in cache.
Variable 'r2_matrix_prediction_planning_over_within' was not found in cache.
Variable 'mse_matrix_prediction_planning_over_within' was not found in cache.
Variable 'Conf_prediction_outcome' was not found in cache.
Variable 'group_names_prediction_outcome' was not found in cache.
Variable 'prediction_outcome_lit_model_state' was not found in cache.
Variable 'correlation_matrix_prediction_outcome_over_mean' was not found in cache.
Variable 'r2_matrix_prediction_outcome_over_mean' was not found in cache.
Variable 'mse_matrix_prediction_outcome_over_mean' was 

AttributeError: 'NoneType' object has no attribute 'seed'

In [171]:
Conf_prediction_planning = load_cache("Conf_prediction_planning")
group_names_prediction_planning = load_cache("group_names_prediction_planning")
prediction_planning_lit_model_state = load_cache("prediction_planning_lit_model_state")
prediction_planning_trainer = load_cache("prediction_planning_trainer")
correlation_matrix_prediction_planning_over_mean = load_cache("correlation_matrix_prediction_planning_over_mean")
r2_matrix_prediction_planning_over_mean = load_cache("r2_matrix_prediction_planning_over_mean")
mse_matrix_prediction_planning_over_mean = load_cache("mse_matrix_prediction_planning_over_mean")
correlation_matrix_prediction_planning_over_within = load_cache("correlation_matrix_prediction_planning_over_within")
r2_matrix_prediction_planning_over_within = load_cache("r2_matrix_prediction_planning_over_within")
mse_matrix_prediction_planning_over_within = load_cache("mse_matrix_prediction_planning_over_within")

Conf_prediction_outcome = load_cache("Conf_prediction_outcome")
group_names_prediction_outcome = load_cache("group_names_prediction_outcome")
prediction_outcome_lit_model_state = load_cache("prediction_outcome_lit_model_state")
prediction_outcome_trainer = load_cache("prediction_outcome_trainer")
correlation_matrix_prediction_outcome_over_mean = load_cache("correlation_matrix_prediction_outcome_over_mean")
r2_matrix_prediction_outcome_over_mean = load_cache("r2_matrix_prediction_outcome_over_mean")
mse_matrix_prediction_outcome_over_mean = load_cache("mse_matrix_prediction_outcome_over_mean")
correlation_matrix_prediction_outcome_over_within = load_cache("correlation_matrix_prediction_outcome_over_within")
r2_matrix_prediction_outcome_over_within = load_cache("correlation_matrix_prediction_outcome_over_within")
mse_matrix_prediction_outcome_over_within = load_cache("mse_matrix_prediction_outcome_over_within")

Conf_outcome_planning = load_cache("Conf_outcome_planning")
group_names_outcome_planning = load_cache("group_names_outcome_planning")
outcome_planning_lit_model_state = load_cache("outcome_planning_lit_model_state")
outcome_planning_trainer = load_cache("outcome_planning_trainer")
correlation_matrix_outcome_planning_over_mean = load_cache("correlation_matrix_outcome_planning_over_mean")
r2_matrix_outcome_planning_over_mean = load_cache("r2_matrix_outcome_planning_over_mean")
mse_matrix_outcome_planning_over_mean = load_cache("mse_matrix_outcome_planning_over_mean")
correlation_matrix_outcome_planning_over_within = load_cache("correlation_matrix_outcome_planning_over_within")
r2_matrix_outcome_planning_over_within = load_cache("r2_matrix_outcome_planning_over_within")
mse_matrix_outcome_planning_over_within = load_cache("mse_matrix_outcome_planning_over_within")

prediction_planning_trainer, prediction_planning_lit_model = build_lit_model(
    Conf_prediction_planning,
    loader_generators,
    "conditional",
    "functional_groups",
    True,
)
prediction_planning_lit_model.load_state_dict(prediction_planning_lit_model_state)

prediction_outcome_trainer, prediction_outcome_lit_model = build_lit_model(
    Conf_prediction_outcome,
    loader_generators,
    "conditional",
    "functional_groups",
    True,
)
prediction_outcome_lit_model.load_state_dict(prediction_outcome_lit_model_state)

outcome_planning_trainer, outcome_planning_lit_model = build_lit_model(
    Conf_outcome_planning,
    loader_generators,
    "conditional",
    "functional_groups",
    True,
)
outcome_planning_lit_model.load_state_dict(outcome_planning_lit_model_state)

Loaded: Conf_prediction_planning
Loaded: group_names_prediction_planning
Loaded: prediction_planning_lit_model_state


AttributeError: Can't get attribute '_refresh_per_optimizer_state' on <module 'torch.cuda.amp.grad_scaler' from 'c:\\Users\\A\\miniconda3\\lib\\site-packages\\torch\\cuda\\amp\\grad_scaler.py'>

### Visualize and Report Results

In [194]:
plot_group_influence_matrix(
    correlation_matrix=correlation_matrix_prediction_planning_over_mean,
    r2_matrix=r2_matrix_prediction_planning_over_mean,
    mse_matrix=mse_matrix_prediction_planning_over_mean,
    group_names=group_names_prediction_planning,
    title="Total Cross-Group Improvements: Reward Prediction and Action Planning",
    file_name="prediction-planning-total-influence-matrix",
    file_path="./plot/model/prediction-planning/group-influence",
)

plot_group_influence_matrix(
    correlation_matrix=correlation_matrix_prediction_planning_over_within,
    r2_matrix=r2_matrix_prediction_planning_over_within,
    mse_matrix=mse_matrix_prediction_planning_over_within,
    group_names=group_names_prediction_planning,
    title="Additional Cross-Group Improvements: Reward Prediction and Action Planning",
    file_name="prediction-planning-additional-influence-matrix",
    file_path="./plot/model/prediction-planning/group-influence",
)

# plot_group_influence_matrix(
#     correlation_matrix=correlation_matrix_prediction_outcome_over_mean,
#     r2_matrix=r2_matrix_prediction_outcome_over_mean,
#     mse_matrix=mse_matrix_prediction_outcome_over_mean,
#     group_names=group_names_prediction_outcome,
#     title="Total Cross-Group Improvements: Reward Prediction and Reward Outcome",
#     filename="prediction-outcome-total-influence-matrix",
#     filepath="./plot/model/prediction-outcome/group-influence",
# )

# plot_group_influence_matrix(
#     correlation_matrix=correlation_matrix_prediction_outcome_over_within,
#     r2_matrix=r2_matrix_prediction_outcome_over_within,
#     mse_matrix=mse_matrix_prediction_outcome_over_within,
#     group_names=group_names_prediction_outcome,
#     title="Additional Cross-Group Improvements: Reward Prediction and Reward Outcome",
#     filename="prediction-outcome-additional-influence-matrix",
#     filepath="./plot/model/prediction-outcome/group-influence",
# )

# plot_group_influence_matrix(
#     correlation_matrix=correlation_matrix_outcome_planning_over_mean,
#     r2_matrix=r2_matrix_outcome_planning_over_mean,
#     mse_matrix=mse_matrix_outcome_planning_over_mean,
#     group_names=group_names_outcome_planning,
#     title="Total Cross-Group Improvements: Reward Outcome and Action Planning",
#     filename="outcome-planning-total-influence-matrix",
#     filepath="./plot/model/outcome-planning/group-influence",
# )

# plot_group_influence_matrix(
#     correlation_matrix=correlation_matrix_outcome_planning_over_within,
#     r2_matrix=r2_matrix_outcome_planning_over_within,
#     mse_matrix=mse_matrix_outcome_planning_over_within,
#     group_names=group_names_outcome_planning,
#     title="Additional Cross-Group Improvements: Reward Outcome and Action Planning",
#     filename="outcome-planning-additional-influence-matrix",
#     filepath="./plot/model/outcome-planning/group-influence",
# )

WindowsPath('plot/model/prediction-planning/group-influence')